# Cycle Time & Idle Detection Analysis

**Purpose:** implement Sections 4.2–4.5 of the paper — DBSCAN stop detection, cycle extraction, phase decomposition, segment classification, and fleet-wide inefficiency quantification (Section D of TASKS.md).

In [ ]:
import sys, warnings
sys.path.append("../..")
warnings.filterwarnings("ignore")

from gps_lib import io_utils, classify, preprocess, zones, stops, cycle_classification, dem, config

## 1. Load + clean

In [ ]:
tracker_list_df = io_utils.load_tracker_list()
tracker_list_df = classify.classify_technic_material_type(tracker_list_df)

gps_raw = io_utils.load_gps_data()
print(f"Raw GPS rows: {len(gps_raw):,}")

merged = preprocess.attach_technic_info(gps_raw, tracker_list_df)
df = preprocess.clean_gps_points(merged, round_n=4)
print(f"After clean: {len(df):,}")

## 2. Motion features + implausible-jump removal

In [ ]:
df = preprocess.add_motion_features(df)
n_jump = int(df["implausible_jump"].sum())
print(f"Implausible pings: {n_jump:,} ({n_jump/len(df):.3%}) — dropping")
df = df[~df["implausible_jump"]].reset_index(drop=True)
df["speed_kmh"].describe(percentiles=[.5,.9,.99])

## 3. Dump trucks only

In [ ]:
dumps = df[df["technic_type"] == "dump"].copy()
print(f"Dump pings: {len(dumps):,}  |  Dump trackers: {dumps['tracker_id'].nunique()}")

## 4. Load zone geometry + DBSCAN eps sizing

In [ ]:
zone_list_df = io_utils.load_zone_list()
zone_list_df = classify.classify_zones(zone_list_df, drop_other=True)
zone_detail_df = io_utils.load_zone_detail()
zones_gdf = zones.build_zone_geodataframe(zone_detail_df, zone_list_df)

diam = zones.zone_diameter_stats(zones_gdf)
diam["diameter_m"].describe(percentiles=[.1,.25,.5,.75,.9]).round(1)

In [ ]:
rec_eps = round(diam["diameter_m"].quantile(0.10) / 2, 1)
print(f"Recommended eps: {rec_eps} m")
print(f"Recommended min_samples: 3  (~60s dwell / 20s ping interval)")

## 5. Assign zone hits to all dump pings

In [ ]:
gps_hits = zones.assign_zone_hit(dumps, zones_gdf)
n_inside = gps_hits["zone_id_hit"].notna().sum()
print(f"Pings inside a zone: {n_inside:,} ({n_inside/len(gps_hits):.1%})")
gps_hits[["tracker_id","get_time","lat","lng","speed","zone_mat_hit","zone_load_hit"]].head(10)

## 6. Segment classification (transit / operating / queuing / unplanned_idle)

In [ ]:
classified = cycle_classification.classify_segments(gps_hits, zones_gdf, speed_threshold=2.0, queue_buffer_m=50.0)
breakdown = cycle_classification.state_time_breakdown(classified, dt_col="dt")
print("\nFleet-wide state-time breakdown (dt-weighted):")
for state, frac in breakdown.items():
    print(f"  {state:<20s}  {frac:.1%}")

In [ ]:
# Monthly trend
import pandas as pd
classified["month"] = pd.to_datetime(classified["get_time"]).dt.to_period("M")
monthly = (
    classified[classified["dt"].notna() & (classified["dt"] > 0)]
    .groupby(["month","state"])["dt"].sum().unstack(fill_value=0)
)
(monthly.div(monthly.sum(axis=1), axis=0) * 100).round(1)

## 7. Cycle extraction + phase decomposition

In [ ]:
cyc_list = cycle_classification.extract_cycles(gps_hits, load_value="load")
cyc_df = cycle_classification.cycles_to_dataframe(cyc_list)
print(f"Total cycles: {len(cyc_df):,}")
print(f"With dump visit: {cyc_df['n_dump_visits'].gt(0).sum():,}")
print(f"Missing dump:    {cyc_df['n_dump_visits'].eq(0).sum():,}")

In [ ]:
# Phase durations in minutes
complete = cyc_df[cyc_df["n_dump_visits"] > 0].copy()
for col in ["load_dwell_sec","haul_to_dump_sec","dump_dwell_sec","haul_to_load_sec","total_sec"]:
    complete[col.replace("sec","min")] = complete[col] / 60
complete[[c for c in complete.columns if c.endswith("_min")]].describe(percentiles=[.25,.5,.75]).round(1)

## 8. DBSCAN stop clustering + unplanned-idle share

In [ ]:
stop_df = stops.filter_stop_pings(gps_hits, speed_threshold=2.0)
print(f"Stop pings: {len(stop_df):,} ({len(stop_df)/len(gps_hits):.1%})")

labeled_stops = stops.cluster_and_label_stops(stop_df, zones_gdf, eps_m=rec_eps, min_samples=3)
n_clustered = (labeled_stops["stop_cluster"] != -1).sum()
n_unplanned = (labeled_stops["zone_mat_hit"] == "unplanned").sum()
print(f"Clustered: {n_clustered:,}  |  In-zone: {n_clustered - n_unplanned:,}  |  Unplanned idle: {n_unplanned:,}")
print(f"Unplanned idle share: {stops.unplanned_idle_share(labeled_stops):.1%}")

## 9. DBSCAN sensitivity sweep

In [ ]:
sweep = stops.dbscan_sensitivity_sweep(
    stop_df, zones_gdf,
    eps_values=[20, rec_eps, 60, 100],
    min_samples_values=[2, 3, 5],
)
sweep

## 10. Top bottleneck load zones (queue dwell)

In [ ]:
queue_pings = classified[classified["state"] == "queuing"].copy()
if not queue_pings.empty:
    near_load = zones.assign_zone_hit(queue_pings, zones_gdf)
    bottlenecks = (
        near_load.groupby("zone_id_hit")["dt"]
        .agg(n_pings="count", total_queue_hr=lambda x: x.sum()/3600)
        .sort_values("total_queue_hr", ascending=False)
        .head(10)
    )
    display(bottlenecks)
else:
    print("No queuing pings detected.")

## 11. DEM grade on haul segments

In [ ]:
DEM_PATH = str(config.DATA_DIR / "dem_merged.tif")
sample_tracker = gps_hits["tracker_id"].iloc[0]
one_day = gps_hits[
    (gps_hits["tracker_id"] == sample_tracker) &
    (pd.to_datetime(gps_hits["get_time"]).dt.date == pd.Timestamp("2025-11-01").date())
].head(200)
graded = dem.add_segment_grade(DEM_PATH, one_day)
graded[["get_time","elevation_m","grade_pct"]].describe(percentiles=[.25,.5,.75]).round(2)